# Week 4 · API, drift monitoring and packaging
This notebook continues Week 3. We make the trained causal models usable by the dashboard, automatically check incoming customer characteristics, and return a budget-respecting plan.

**Simple example:** A shop trained its model on customers spending around $100 previously. If tomorrow's upload contains mostly customers with $10,000 histories, a warning asks us to investigate before trusting the predictions.

Run notebooks 02 and 05 first, then 06 for the audit. Use the repository root as the working directory and the project Python 3.12 kernel. Operational API modules are shared with this notebook so there are no two different versions of the algorithm.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from fastapi.testclient import TestClient
from api.main import app
from api.service import BASELINE, Engine, engineer, drift
ROOT = Path.cwd()
assert (ROOT / "api/main.py").exists(), "Open the repository root before running."
OUTPUT = ROOT / "results/week4"
OUTPUT.mkdir(parents=True, exist_ok=True)
client = TestClient(app)
engine = Engine()
print("Training reference rows:", len(engine.reference))
print("Model fingerprint:", engine.model_id)


/Users/sharath/Documents/ChatGPT/econ_double_ml_project/project-week4/.runtime-deps/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


/Users/sharath/economl_project-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training reference rows: 44800
Model fingerprint: c5b6e25bcf2cc60a008bbb3c004e21d3394cb6227078a564d3a66b3e0cf56867


## 1. Check that the API reproduces notebook 05
We score 200 held-out customers using their pre-campaign information only. We compare all six effects against notebook 05: men's and women's email effects on visits, purchases and spending. No model is retrained on these customers.


In [2]:
saved = pd.read_csv(ROOT / "results/double_ml/test_scores.csv").head(200)
clean = pd.read_csv(ROOT / "data/processed/causal_clean.csv").set_index("customer_id")
sample = clean.loc[saved.customer_id].reset_index()[["customer_id"] + BASELINE]
customers = sample.to_dict("records")
response = client.post("/api/score", json={"customers": customers})
assert response.status_code == 200, response.text
actual = pd.DataFrame(response.json()["rows"])
errors = {}
for outcome in ["visit", "conversion", "spend"]:
    for arm in ["mens", "womens"]:
        name = f"{outcome}_{arm}"
        errors[name] = float(np.max(np.abs(actual[name].to_numpy() - saved[name].to_numpy())))
assert max(errors.values()) < 1e-9, "API and notebook predictions differ."
display(pd.Series(errors, name="maximum_absolute_prediction_difference"))


visit_mens           1.526557e-16
visit_womens         1.804112e-16
conversion_mens      9.887924e-17
conversion_womens    9.909608e-17
spend_mens           1.776357e-15
spend_womens         1.554312e-15
Name: maximum_absolute_prediction_difference, dtype: float64

## 2. Automatic drift screening
Numeric characteristics use the KS distance; category proportions use total variation distance. Both are zero for identical distributions. A distance of 0.1 triggers review. This is an explicit screening heuristic, not a statistical proof or a causal-effect test. New categories also trigger review. Batches below 100 customers receive an insufficient-data status.

We test an unchanged reference sample, then deliberately change history and channel. These artificial changes are only tests; they are not findings about actual new customers.


In [3]:
unchanged = drift(engine.reference, engine.reference.copy())
assert unchanged["status"] == "no_warning"
shifted = sample.copy()
shifted["history"] = 100000.0
shifted["channel"] = "Unknown demo channel"
changed = client.post("/api/drift", json={"customers": shifted.to_dict("records")}).json()
assert changed["status"] == "review"
small = client.post("/api/drift", json={"customers": customers[:2]}).json()
assert small["status"] == "insufficient_data"
display(pd.DataFrame(changed["metrics"]))
print("Unchanged:", unchanged["status"], "| Deliberately shifted:", changed["status"], "| Small batch:", small["status"])


,feature,metric,distance,threshold,unknown_fraction,warning
0,recency,KS distance,0.033192,0.1,0.0,False
1,history,KS distance,1.000000,0.1,0.0,True
2,mens,Total variation distance,0.007188,0.1,0.0,False
3,womens,Total variation distance,0.043504,0.1,0.0,False
4,newbie,Total variation distance,0.048080,0.1,0.0,False
5,zip_code,Total variation distance,0.022478,0.1,0.0,False
6,channel,Total variation distance,1.000000,0.1,1.0,True


Unchanged: no_warning | Deliberately shifted: review | Small batch: insufficient_data


## 3. One request: score, check drift and recommend
With $1 and a $0.10 cost per email, we may contact at most ten customers. The same SciPy optimizer as notebook 07 chooses each customer's better positive spending effect. Both email types have the same assumed cost. Contact-cost avoidance versus emailing everyone is not proof of increased profit.


In [4]:
campaign = client.post("/api/recommend", json={"customers": customers, "budget": 1.0, "email_cost": 0.10})
assert campaign.status_code == 200, campaign.text
result = campaign.json()
assert result["allocation"]["contacted"] <= 10
assert result["allocation"]["allocated_cost"] <= 1.0
assert "drift" in result
print(result["plain_summary"])
display(pd.DataFrame(result["rows"])[["customer_id","recommended_action","allocated_cost","predicted_incremental_revenue"]].head(20))


Contact 10 of 200 customers, spending $1.00. This uses $19.00 less contact cost than emailing everyone. Predicted additional revenue is $22.82; it is not measured sales.


,customer_id,recommended_action,allocated_cost,predicted_incremental_revenue
0,row_000001,No E-Mail,0.0,0.000000
1,row_000005,No E-Mail,0.0,0.000000
2,row_000006,No E-Mail,0.0,0.000000
3,row_000015,No E-Mail,0.0,0.000000
4,row_000017,No E-Mail,0.0,0.000000
5,row_000023,No E-Mail,0.0,0.000000
6,row_000031,No E-Mail,0.0,0.000000
7,row_000032,No E-Mail,0.0,0.000000
8,row_000034,No E-Mail,0.0,0.000000
9,row_000035,Womens E-Mail,0.1,2.622901


## 4. Reject mistakes and save the verification report
Duplicate IDs, invalid histories, campaign outcomes accidentally included as predictors, and fractional-cent budgets must be rejected. This protects the meaning of each prediction and the budget.


In [5]:
for invalid in [[customers[0], customers[0]], [dict(customers[0], history=-1)], [dict(customers[0], visit=1)]]:
    assert client.post("/api/score", json={"customers": invalid}).status_code == 422
assert client.post("/api/recommend", json={"customers": customers[:1], "budget": 0.001}).status_code == 422
report = {"model_id": engine.model_id, "reference_rows": len(engine.reference),
          "prediction_parity_errors": errors, "unchanged_drift": unchanged["status"],
          "shifted_drift": changed["status"], "small_batch": small["status"],
          "campaign": result["allocation"], "validation_checks": "passed"}
(OUTPUT / "verification.json").write_text(json.dumps(report, indent=2))
print("Saved results/week4/verification.json")


Saved results/week4/verification.json


## How to explain this in a viva
“Week 4 turns my analysis into a usable local application. The API receives customer characteristics, applies exactly the same trained preprocessing and causal models, checks for changes in the incoming data, and uses the Week 3 optimizer to recommend emails within a budget. The dashboard explains the recommendation and lets users download it.”

**Limits:** Drift warnings do not prove hidden confounding or a changed causal effect. Passing the audit does not prove that all hidden confounders are absent. Spending rankings remain uncertain and the women's-email subset audit needs review. This local demonstration has no public-host authentication or production monitoring service; keep it on localhost. The model predicts email effects, not $10-versus-$20 discount effects.

See `docs/WEEK_4_WRITEUP.md` and `docs/API_REFERENCE.md` for setup and operation.
